In [1]:
import langchain_openai
import os
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage, SystemMessage
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
import numpy as np
import pandas as pd

In [3]:
from langchain_openai import OpenAIEmbeddings

In [4]:
from langchain_chroma import Chroma

In [5]:
from langchain.schema import Document

In [6]:
import random

In [7]:
#chatModel = ChatOpenAI(model="gpt-4o-mini-2024-07-18")
chatModel = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0.8  # Adjust the temperature as needed
)

In [8]:
systemPrompt="""
You are a helpful assistant that generates realistic customer support tickets. The tickets should reflect the sentiment specified by the user. The possible sentiment categories are:

- Strong Negative
- Mild Negative
- Neutral
- Mild Positive
- Strong Positive

For each ticket, ensure that the content matches the given sentiment category. Vary the topics, language styles, and lengths to simulate real-world customer communications. Avoid using offensive language or disallowed content. Do not include any personal identifiable information (PII).

When given a sentiment category, generate a customer support ticket that exemplifies that sentiment.
"""

create different user prompts for 5 sentiments

In [9]:
userPromptStrongNegative="Generate a customer support ticket expressing strong negative sentiment. The customer is extremely unhappy about a severe issue with a product or service failure. The complaint should be intense and convey frustration and disappointment."
userPromptMildNegative="Generate a customer support ticket expressing mild negative sentiment. The customer is somewhat dissatisfied with an aspect of the product or service. The complaint should be moderate and suggest disappointment without extreme language."
userPromptNeutral="Generate a customer support ticket with neutral sentiment. The customer is requesting information or assistance without expressing positive or negative emotions. The message should be straightforward and objective."
userPromptMildPositive="Generate a customer support ticket expressing mild positive sentiment. The customer is generally satisfied and perhaps offers a small compliment but may include a suggestion for improvement. The tone should be pleasant and appreciative."
userPromptStrongPositive="Generate a customer support ticket expressing strong positive sentiment. The customer is extremely happy with the product or service and is providing enthusiastic praise. The message should convey excitement and high satisfaction."

In [ ]:
#LabelSentiments

In [10]:
LabelStrongNegative="Strong Negative"
LabelMildNegative="Mild Negative"
LabelNeutral="Neutral"
LabelMildPositive="Mild Positive"
LabelStrongPositive="Strong Positive"

In [ ]:
#we will store all data in a variable

In [11]:
all_tickets = []

In [ ]:
#we will create 10 examples for each sentiment

In [12]:
example_count=10

In [ ]:
#First create 10 Strong Negative, and store them with numbered Labels

In [13]:
messages = [
    ("system", systemPrompt),
    ("human",userPromptStrongNegative),
]

In [14]:
for _ in range(example_count):
    response = chatModel.invoke(messages)
    all_tickets.append({'Ticket Text':response.content, 'Sentiment':LabelStrongNegative})

In [15]:
all_tickets

[{'Ticket Text': '**Ticket ID:** 8392023\n\n**Subject:** Urgent: Extremely Disappointed with Product Failure\n\n**Description:**\n\nI am writing to express my utter frustration and disappointment with the [Product Name] that I purchased from your company. I expected a high-quality item based on your marketing and past reviews, but what I received was nothing short of a disaster.\n\nAfter just two weeks of use, the product completely failed. It stopped functioning without any warning, and I have followed all the maintenance instructions provided. This is unacceptable for a product of this price range! I’ve tried reaching out to your customer service multiple times, but I’ve either been left on hold for ages or received vague responses that do not address the issue at all.\n\nI feel completely let down by your brand. I had high hopes for this purchase, and now I’m stuck with a useless item and no real support. I demand an immediate resolution—either a full refund or a replacement that ac

In [ ]:
#Create 10 Mild Negative, and store them with numbered Labels

In [16]:
messages = [
    ("system", systemPrompt),
    ("human",userPromptMildNegative),
]

In [17]:
for _ in range(example_count):
    response = chatModel.invoke(messages)
    all_tickets.append({'Ticket Text':response.content, 'Sentiment':LabelMildNegative})

In [ ]:
#Create 10 Neutral, and store them with numbered Labels

In [18]:
messages = [
    ("system", systemPrompt),
    ("human",userPromptNeutral),
]

In [19]:
for _ in range(example_count):
    response = chatModel.invoke(messages)
    all_tickets.append({'Ticket Text':response.content, 'Sentiment':LabelNeutral})

In [ ]:
#Create 10 MildPositive, and store them with numbered Labels

In [20]:
messages = [
    ("system", systemPrompt),
    ("human",userPromptMildPositive),
]

In [21]:
for _ in range(example_count):
    response = chatModel.invoke(messages)
    all_tickets.append({'Ticket Text':response.content, 'Sentiment':LabelMildPositive})

In [ ]:
#Create 10 StrongPositive, and store them with numbered Labels

In [22]:
messages = [
    ("system", systemPrompt),
    ("human",userPromptStrongPositive),
]

In [23]:
for _ in range(example_count):
    response = chatModel.invoke(messages)
    all_tickets.append({'Ticket Text':response.content, 'Sentiment':LabelStrongPositive})

In [24]:
all_tickets

[{'Ticket Text': '**Ticket ID:** 8392023\n\n**Subject:** Urgent: Extremely Disappointed with Product Failure\n\n**Description:**\n\nI am writing to express my utter frustration and disappointment with the [Product Name] that I purchased from your company. I expected a high-quality item based on your marketing and past reviews, but what I received was nothing short of a disaster.\n\nAfter just two weeks of use, the product completely failed. It stopped functioning without any warning, and I have followed all the maintenance instructions provided. This is unacceptable for a product of this price range! I’ve tried reaching out to your customer service multiple times, but I’ve either been left on hold for ages or received vague responses that do not address the issue at all.\n\nI feel completely let down by your brand. I had high hopes for this purchase, and now I’m stuck with a useless item and no real support. I demand an immediate resolution—either a full refund or a replacement that ac

In [26]:
print(len(all_tickets))

50


We can use the similarity_search_with_score method of vector DBs to prevent duplicate records. We add the first ticket to the vector DB. If the score in the similarity search for a new ticket is very low, it means the data is extremely similar to an existing ticket, and we ignore it.

In [27]:
embeddings = OpenAIEmbeddings()

In [28]:
vector_db = Chroma(embedding_function=embeddings)
documents = [Document(page_content="First data", metadata={})]
vector_db.add_documents(documents)

['6c97705d-2413-4681-a576-343e5883435b']

In [29]:
unique_tickets=[]

if the similarity <0.1 we assume this is very similar and ignore this data

In [30]:
for ticket in all_tickets:
    similarity = vector_db.similarity_search_with_score(ticket['Ticket Text'],k=1)
    if similarity[0][1]>0.1:
        documents = [Document(page_content=ticket['Ticket Text'], metadata={})]
        vector_db.add_documents(documents)
        unique_tickets.append(ticket)

In [32]:
len(unique_tickets)

38

In [33]:
random.shuffle(unique_tickets)

In [34]:
total_size = len(unique_tickets)
    
# calculate sizes
test_size = int(total_size * 0.10)
val_size = int(total_size * 0.10)
train_size = total_size - test_size - val_size  # %80

# Split set
test_set = unique_tickets[:test_size]
val_set = unique_tickets[test_size:test_size+val_size]
train_set = unique_tickets[test_size+val_size:]

In [35]:
test_df = pd.DataFrame(test_set)
test_df.to_csv("test.csv", index=False)

validate_df = pd.DataFrame(val_set)
validate_df.to_csv("validate.csv", index=False)

train_df = pd.DataFrame(train_set)
train_df.to_csv("train.csv", index=False)

Summary

In generating the dataset, I chose to create 10 examples for each of five sentiment categories (Strong Negative, Mild Negative, Neutral, Mild Positive, Strong Positive), resulting in a balanced dataset that captures a range of emotional expressions. This size provides sufficient variety while keeping the dataset manageable for computational efficiency. To ensure data quality and uniqueness, I implemented deduplication using a vector similarity search, which filters out overly similar examples by setting a similarity threshold. I also used a temperature setting of 0.8 during text generation to introduce controlled randomness, producing diverse but relevant responses. Finally, I split the dataset into training, validation, and test sets (80/10/10).


To prevent overfitting (during training in the 03_Train.jpynb document), I used the Early Stopping method. I stop the training once the validation loss starts to increase or become equal, indicating potential overfitting.